## Objective
The objective hee is to train a model to distinguish between mountain and glecier.


https://www.kaggle.com/code/injin1992/starter-intel-image-classification-2013d5e3-3/input


In [1]:
# Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Read the image files
# import os
# import cv2
# import numpy as np

# for dirname, _, filenames in os.walk('/content/drive/MyDrive/edurekaai/_data/mountains/'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

In [2]:
# ===========================
# 1) Imports
# ===========================
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import pandas as pd

# ===========================
# 2) Paths and parameters
# ===========================
TRAIN_DIR = '/content/drive/MyDrive/edurekaai/_data/nature/seg_train/'
TEST_DIR  = '/content/drive/MyDrive/edurekaai/_data/nature/seg_test/'
IMG_SIZE  = (150, 150)
BATCH     = 32
VAL_SPLIT = 0.2
EPOCHS    = 30

# ===========================
# 3) Data Generators
# ===========================
train_datagen = ImageDataGenerator(
    # Image pixel values are between 0...255.
    # rescale=1.0/255 devides the pixels by 255. So it value ranges from 0...1.
    rescale=1.0/255,
    # validation_split=0.2 splits the input folder data into 80:20 for train:test.
    validation_split=VAL_SPLIT,
    rotation_range=15,
    width_shift_range=0.08,
    height_shift_range=0.08,
    zoom_range=0.12,
    shear_range=0.08,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1.0/255)

# Train generator
## class_mode=binary means only 2 class of images. class_mode=categorical means multi class images.
## sub_folder names (like mountain, street, sea,... are auto-picked up as the class of the images)
## target_size=(150,150) resizes the images to 150 x 150 pixel images.
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH,
    class_mode='categorical',
    subset='training'
)

# Validation generator (from training set split)
val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH,
    class_mode='categorical',
    subset='validation'
)

# Test generator
test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH,
    class_mode='categorical',
    shuffle=False
)

num_classes = train_generator.num_classes
print("Classes:", train_generator.class_indices)

# ===========================
# 4) Model Builder Functions
# ===========================
def build_simple_cnn(input_shape=(150,150,3), num_classes=6):
    model = models.Sequential([
        layers.Conv2D(filters=32, kernel_size=(3,3), activation='relu', input_shape=input_shape),
        layers.MaxPooling2D((2,2)),

        layers.Conv2D(filters=64, kernel_size=(3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),

        layers.Conv2D(filters=128, kernel_size=(3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),

        layers.Flatten(),

        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),

        layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

def build_vgg_like(input_shape=(150,150,3), num_classes=6):
    model = models.Sequential([
        layers.Conv2D(64, (3,3), activation='relu', padding='same', input_shape=input_shape),

        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),

        layers.Conv2D(128, (3,3), activation='relu', padding='same'),

        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),

        layers.Conv2D(256, (3,3), activation='relu', padding='same'),

        layers.Conv2D(256, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),

        layers.Flatten(),

        layers.Dense(512, activation='relu'),
        layers.Dropout(0.5),

        layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# ===========================
# 5) Callbacks Function
# ===========================
def get_callbacks(model_name):
    return [
        callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(factor=0.2, patience=3),
        callbacks.ModelCheckpoint(f"{model_name}.h5", save_best_only=True)
    ]

# ===========================
# 6) Train and Evaluate Multiple Models
# ===========================
models_to_train = {
    "simple_cnn": build_simple_cnn,
    "vgg_like": build_vgg_like
}

results = []

for name, build_fn in models_to_train.items():
    print(f"\n=== Training {name} ===")
    model = build_fn(input_shape=(IMG_SIZE[0], IMG_SIZE[1],3), num_classes=num_classes)

    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=EPOCHS,
        callbacks=get_callbacks(name)
    )

    test_loss, test_acc = model.evaluate(test_generator)
    print(f"{name} Test Accuracy: {test_acc:.4f}")

    results.append({
        "model": name,
        "test_accuracy": test_acc,
        "test_loss": test_loss,
        "history": history
    })

# ===========================
# 7) Compare Models
# ===========================
df_results = pd.DataFrame([{"Model": r["model"], "Test Accuracy": r["test_accuracy"], "Test Loss": r["test_loss"]} for r in results])
print("\nModel Comparison:")
print(df_results)

# Optional: plot training histories
for r in results:
    plt.plot(r["history"].history["val_accuracy"], label=f"{r['model']} val_acc")
plt.title("Validation Accuracy Comparison")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()


Found 4813 images belonging to 6 classes.
Found 1201 images belonging to 6 classes.
Found 437 images belonging to 6 classes.
Classes: {'buildings': 0, 'forest': 1, 'glacier': 2, 'mountain': 3, 'sea': 4, 'street': 5}

=== Training simple_cnn ===


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
 12/151 ━━━━━━━━━━━━━━━━━━━━ 45:53 20s/step - accuracy: 0.3230 - loss: 1.5350

KeyboardInterrupt: 